# Session 10 — DQN lab, an RL-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 5 · Practice session (about 80 minutes)**

I am Salih Tutun. I talk. We bet as a class. Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Neighbor, two voices, then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

**Family this session: RL-based agent.** Compare the learned policy with a base-stock rule. Do not describe this network as an LLM-based agent.

![Deep Q-network](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/dqn.png)

## Run of show
- 10 min — Baseline first. A neural net that loses to "order up to 2" is not a win.
- 55 min — Finish the TD target and the weight update. Compare DQN to the base-stock rule.
- 15 min — Deploy / do not deploy, and the human override.

Same supermarket Home-and-lifestyle shelf. Demand is sampled from the real invoices. State features are scaled stock and the demand just observed. Actions are still order 0, 1, or 2.

## The update, before you code it

The baseline is **base-stock**: order enough to bring stock back to 2, and never order more than 2. A network that loses to that rule is not a deployable agent. Compare averages, not a single lucky day.

**One step of DQN**, using the ideas from Session 9:

1. Turn the observation into features `x` (stock and the demand you just saw, each scaled into a small range).
2. The network's forward pass returns three numbers, Q for order 0, 1, and 2. Act ε-greedy on those numbers.
3. Store `(x, action, pay, next_x)` in a memory list. That is experience replay: you will learn from a past tuple, not only from the order you just placed.
4. The TD target is `pay + GAMMA * max Q(next_x)`. This lab may compute that max on the same network you are training. Say in the debrief why Session 9's frozen target copy is safer: the target stops moving on every step.
5. Move only the output for the action you took. Copy the Q vector, set `grad_q[action] = Q(x, action) - target`, and leave the other two entries at 0. The weight step is `w2 = w2 - lr * outer(hidden, grad_q)`. The same error updates the output bias. The other two actions do not move on this step.

A short run often **loses** to base-stock. That is a result. The call in the room is do not deploy unless you can name the human override.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I explain. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor. Two people share. Then type `YOU TYPE HERE`. |
| **TODO** / `YOU WRITE THIS` | You write the function. Search for `TODO`. We stay until every line says PASS. |
| **RUN WITH ME** | Already written. Run it. Do not type in that cell. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** — and one line for your project framework. |


## Your work — TODO (you write this in class)

Search for `TODO`. Finish `learn` and the replay sample. Then compare the average with the base-stock cell before anyone says deploy.


In [ ]:
import numpy as np

from pathlib import Path
from collections import Counter, defaultdict
import csv

def course_data(name):
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(name)


def empirical_demand_p():
    """Bin daily Home-and-lifestyle quantity into {0, 1, 2}."""
    daily = defaultdict(int)
    with course_data("supermarket-sales.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product line") or "").strip() != "Home and lifestyle":
                continue
            daily[row["Date"]] += int(float(row["Quantity"]))
    totals = sorted(daily.values())
    q1, q2 = totals[len(totals) // 3], totals[(2 * len(totals)) // 3]
    bins = [0 if qty <= q1 else 1 if qty <= q2 else 2 for qty in totals]
    counts = Counter(bins)
    n = max(len(bins), 1)
    return {level: counts[level] / n for level in (0, 1, 2)}


DEMAND_P = empirical_demand_p()
DEMAND_LEVELS = list(DEMAND_P.keys())
DEMAND_PROBS = [DEMAND_P[k] for k in DEMAND_LEVELS]
print("Empirical P(demand) from Home-and-lifestyle invoices:", {k: round(v, 3) for k, v in DEMAND_P.items()})

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5
RNG = np.random.default_rng(11)


def step(stock, order):
    demand = int(RNG.choice(DEMAND_LEVELS, p=DEMAND_PROBS))
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    missed = max(0, demand - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay, demand


def features(stock, demand):
    return np.array([stock / CAPACITY, demand / 2.0])


class TinyQ:
    def __init__(self, rng):
        self.w1 = rng.normal(0, 0.2, size=(2, 8))
        self.b1 = np.zeros(8)
        self.w2 = rng.normal(0, 0.2, size=(8, 3))
        self.b2 = np.zeros(3)

    def forward(self, x):
        self.x = np.asarray(x, dtype=float)
        self.z1 = self.x @ self.w1 + self.b1
        self.h = np.maximum(0, self.z1)
        return self.h @ self.w2 + self.b2

    def learn(self, x, action, target, lr=0.05):
        """TODO: move w2/b2 (and, if you have time, w1/b1) so Q(x, action) gets closer to target.

        Hint: q = forward(x). You only want to change the output for `action`.
        grad_q = q.copy(); grad_q[action] = q[action] - target
        Then dw2 = outer(h, grad_q); subtract lr * dw2 from w2.
        """
        q = self.forward(x)
        # TODO
        return q


def base_stock_return(days=300):
    stock, demand, total = 1, 1, 0.0
    for _ in range(days):
        order = max(0, min(2, 2 - stock))
        stock, pay, demand = step(stock, order)
        total += pay
    return total


print("base-stock return over 300 days", round(base_stock_return(), 2))


In [ ]:
def train(steps=1500, epsilon=0.20):
    online = TinyQ(np.random.default_rng(3))
    memory = []
    stock, demand = 1, 1
    recent = []
    for t in range(steps):
        x = features(stock, demand)
        q = online.forward(x)
        action = int(RNG.integers(0, 3)) if RNG.random() < epsilon else int(np.argmax(q))
        ending, pay, new_demand = step(stock, action)
        memory.append((x, action, pay, features(ending, new_demand)))
        # TODO: sample one memory tuple and call online.learn with the TD target
        # gamma * max(online.forward(next_x)) + pay.
        # Using the online net as the target is acceptable for this lab; say in the debrief why a frozen copy is safer.
        recent.append(pay)
        stock, demand = ending, new_demand
    return online, float(np.mean(recent[-100:]))


net, avg_pay = train()
print("average pay of last 100 steps", round(avg_pay, 3))
print("Fill the TODO. Then compare this average with the base-stock cell before you recommend deployment.")


## Written close (in class)

1. Did DQN beat, match, or lose to base-stock on your run?
2. One failure you would watch in a longer run (divergence, no exploration, or a reward that ignores stockouts).
3. The human override you would require if this touched a real purchase order.


## Fill this with me — type this in class

**STOP. Do not scroll.** Neighbor 20s. Two voices. Bet before you trust the printout. Then point at the two numbers and say whether you would sign a purchase order.

1. Type `beats_base_stock` as `"yes"` or `"no"`.
2. After the numbers, type `our_call` as `"deploy"` or `"do not deploy"`.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. "yes" or "no" — will this short run beat base-stock?
beats_base_stock = ""  # YOU TYPE HERE first
# After the numbers. "deploy" or "do not deploy".
our_call = ""  # YOU TYPE HERE after the numbers
print("Bet:", beats_base_stock or "(bet first)")
print("Call:", our_call or "(after the numbers)")


## Add this to your framework — STOP. Do not scroll.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep.

Hint: losing to base-stock is allowed if you say so. Do not deploy an RL policy without a human override. Language exceptions still need the LLM family plus a reviewer.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
